# Comparación de modelos para aprobación de crédito

Parcial práctico — Machine Learning, Universidad Libre.
Dataset individual: `aprobacion_credito_Arias_Becerra_Johel_Santiago_3144.csv` (1.000 solicitudes).

Trabajo en riesgo: hay que recomendar, con evidencia, un clasificador para aprobar o rechazar crédito entre un **árbol de decisión**, una **regresión logística** y una **SVM**.

Reglas de este notebook (las de clase y las del diseño):

- Un solo split: `test_size=0.25`, `random_state=11`, `stratify=y`.
- Escalamos SVM y logística con `StandardScaler` **dentro** de un `Pipeline` (el scaler se ajusta solo con train). El árbol no se escala.
- Evaluamos solo en test: accuracy, precisión, sensibilidad (recall), especificidad y AUC.
- Criterio de recomendación, declarado **antes** de ver la tabla: mirar recall, especificidad y AUC. Un modelo domina si gana al menos 2 de esas 3 por ≥ 0.03 frente a cada rival. Si el desempeño está cercano, priorizamos interpretabilidad (árbol > logística > SVM). No elegimos por accuracy: el 73.3% de las solicitudes históricas fueron rechazadas.

## Contexto


## Carga y EDA

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

COLUMNAS = [
    "ingreso_mensual_kcop",
    "score_buro",
    "antiguedad_laboral_anios",
    "num_creditos_activos",
    "dti",
    "aprobado",
]

NOMBRE_CSV = "aprobacion_credito_Arias_Becerra_Johel_Santiago_3144.csv"
RUTA_ESPERADA = f"data/{NOMBRE_CSV}"


def ruta_dataset() -> Path:
    candidatos = [
        Path("data") / NOMBRE_CSV,
        Path("../data") / NOMBRE_CSV,
        Path.cwd() / "data" / NOMBRE_CSV,
        Path.cwd().parent / "data" / NOMBRE_CSV,
    ]
    for path in candidatos:
        if path.is_file():
            return path.resolve()
    raise FileNotFoundError(
        f"No se encontró el dataset. Ruta esperada: {RUTA_ESPERADA}"
    )


def cargar_dataset(path: Path | None = None) -> pd.DataFrame:
    destino = Path(path) if path is not None else ruta_dataset()
    if not destino.is_file():
        raise FileNotFoundError(
            f"No se encontró el dataset. Ruta esperada: {RUTA_ESPERADA}"
        )
    df = pd.read_csv(destino)
    if list(df.columns) != COLUMNAS:
        raise ValueError(
            f"columnas distintas a las esperadas {COLUMNAS}; recibidas {list(df.columns)}"
        )
    if len(df) != 1000:
        raise ValueError(f"se esperaban 1000 filas; hay {len(df)}")
    if df.isna().any().any():
        raise ValueError("el dataset no debe tener nulos")
    if not set(df["aprobado"].unique()) <= {0, 1}:
        raise ValueError("aprobado debe ser 0 o 1")
    return df


def partir_train_test(df: pd.DataFrame):
    X = df.drop(columns="aprobado")
    y = df["aprobado"]
    return train_test_split(X, y, test_size=0.25, random_state=11, stratify=y)


def metricas_prueba(y_true, y_pred, y_score) -> dict[str, float]:
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    if tn + fp == 0:
        raise ValueError("especificidad indefinida: no hay negativos reales")
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "especificidad": float(tn / (tn + fp)),
        "auc": float(roc_auc_score(y_true, y_score)),
    }


def construir_modelos() -> dict:
    return {
        "Árbol": DecisionTreeClassifier(max_depth=4, random_state=11),
        "Regresión logística": Pipeline(
            [
                ("scaler", StandardScaler()),
                ("clf", LogisticRegression(max_iter=1000, random_state=11)),
            ]
        ),
        "SVM": Pipeline(
            [
                ("scaler", StandardScaler()),
                ("clf", SVC(kernel="rbf", probability=True, random_state=11)),
            ]
        ),
    }


In [ ]:
df = cargar_dataset()
print(df.shape)
print(df.dtypes)
print(df.isna().sum())
print(df["aprobado"].value_counts())
print("tasa de aprobación", round(df["aprobado"].mean(), 3))

In [ ]:
print(df.describe().T)
print(df.groupby("aprobado")[["ingreso_mensual_kcop", "score_buro", "antiguedad_laboral_anios", "num_creditos_activos", "dti"]].mean())
print(df.corr(numeric_only=True)["aprobado"].sort_values(ascending=False))

Hay **267 aprobados (26.7%)** y 733 rechazados. Sin nulos.

El score de buró es la señal más clara (media ~701 en aprobados vs ~628 en rechazados). El DTI también separa (más bajo entre aprobados). El ingreso mensual casi no cambia entre clases. Este notebook no fabrica variables nuevas: usamos las cinco columnas tal cual.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
conteo = df["aprobado"].value_counts().sort_index()
axes[0].bar(["Rechazado (0)", "Aprobado (1)"], conteo.values)
axes[0].set_title("Desbalance de la decisión histórica")
axes[0].set_ylabel("Solicitudes")

df.boxplot(column="score_buro", by="aprobado", ax=axes[1])
axes[1].set_title("Score de buró por decisión histórica")
axes[1].set_xlabel("aprobado")
axes[1].set_ylabel("score_buro")
plt.suptitle("")
plt.tight_layout()
plt.show()

## Split

Mismo corte que en clase. `stratify=y` conserva el 26.7% de aprobados en train y test.

In [ ]:
Xtr, Xte, ytr, yte = partir_train_test(df)
print(Xtr.shape, Xte.shape)
print("aprobados train", int(ytr.sum()), "tasa", round(ytr.mean(), 4))
print("aprobados test", int(yte.sum()), "tasa", round(yte.mean(), 4))

## Entrenamiento de los 3 modelos

- Árbol: `max_depth=4` para no memorizar train y poder explicar las reglas. No lleva scaler: los cortes no dependen de la unidad de medida.
- Logística y SVM: `Pipeline` con `StandardScaler`. La SVM es un modelo de distancia; la logística también se beneficia del escalado. `probability=True` en la SVM permite el AUC con `predict_proba` (en scikit-learn reciente puede salir un aviso de deprecación; lo dejamos porque es el contrato de este ejercicio).

In [ ]:
modelos = construir_modelos()
for nombre, modelo in modelos.items():
    modelo.fit(Xtr, ytr)
    print("entrenado:", nombre)

## Métricas en test

Clase positiva = 1 (aprobado). Especificidad = TN / (TN + FP): de los que el histórico rechazó, cuántos rechazamos.

In [ ]:
import pandas as pd

filas = []
predicciones = {}
for nombre, modelo in modelos.items():
    pred = modelo.predict(Xte)
    score = modelo.predict_proba(Xte)[:, 1]
    predicciones[nombre] = pred
    m = metricas_prueba(yte, pred, score)
    filas.append({"modelo": nombre, **m})

tabla = pd.DataFrame(filas).set_index("modelo")
tabla_3 = tabla.round(3)
tabla_3

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (nombre, pred) in zip(axes, predicciones.items()):
    cm = confusion_matrix(yte, pred, labels=[0, 1])
    print(nombre, "TN FP FN TP", cm.ravel().tolist(), "suma", int(cm.sum()))
    ConfusionMatrixDisplay(cm, display_labels=["Rechazo (0)", "Aprobado (1)"]).plot(ax=ax, colorbar=False)
    ax.set_title(nombre)
plt.tight_layout()
plt.show()